# 07 - Observability

Every notebook in this pipeline (01-06) logs through `RunLogger` (`utils/observability/run_logger.py`) - each run writes a structured JSON record to `datasets/reports/runs/` with per-stage timing, status, and metrics. This notebook reads those back and renders the pipeline's actual execution history, plus the architecture/dataflow diagram (`utils/observability/diagram.py`).

In [1]:
import os
import sys

# Portable project-root resolution (no machine-specific hardcoded path) -
# walk up from the current working directory until pyproject.toml is found.
# One statement on purpose: ruff/pycodestyle's E402 ("imports not at top")
# specifically exempts a lone sys.path.insert(...) call, not a multi-
# statement block before it.
sys.path.insert(0, next(
    d for d in (
        os.path.abspath(os.path.join(os.getcwd(), *([os.pardir] * i)))
        for i in range(8)
    )
    if os.path.exists(os.path.join(d, "pyproject.toml"))
))

import pandas as pd

from utils.observability.diagram import ARCHITECTURE_DIAGRAM
from utils.observability.run_logger import load_recent_runs

_root = sys.path[0]


## Architecture / dataflow diagram

Rendered from `utils/observability/diagram.py` (Mermaid) - Jupyter renders Mermaid natively when the `mermaid`/`ipymermaid` extension is available; the raw source always prints below regardless, and the same string is embedded in `docs/architecture.md`.

In [2]:
print(ARCHITECTURE_DIAGRAM)

flowchart TB
    subgraph SOURCES["Cloud Sources"]
        S3["AWS S3"]
        ADLS["Azure ADLS Gen2"]
        GCS["Google Cloud Storage"]
        HTTP["Public HTTPS bucket"]
    end

    subgraph INGEST["utils/ingestion"]
        CONN["IngestionConnector\n(s3 / adls / gcs / http / local)"]
    end

    RAW[("datasets/raw/\nlocal, analysis-ready")]

    subgraph PROCESS["notebooks/02_process\n+ utils/processing"]
        CAL["Lee speckle filter\n+ linear-to-dB"]
    end

    subgraph FEAT["notebooks/03_feature_engineering"]
        TEX["Texture / polarimetric\nfeatures + NaN masking"]
        VEC["Per-pixel feature vectors\n(vv/vh/ratio/diff/local stats)"]
    end

    subgraph CLASSICAL["Classical ML"]
        RF["Random Forest\n(notebook 04, pixel-wise)"]
        UNET["U-Net\n(notebook 09, patch-wise)"]
    end

    subgraph QUANTUM["Quantum ML - utils/qml\n(FORCE_SIMULATION default)"]
        IBM["IBM Quantum Runtime\n(QiskitRuntimeService / AerSimulator)"]
        QKERNEL["Quantum

## Recent pipeline runs

One row per stage, across every notebook run so far. Run notebooks 01-06 first if this is empty.

In [3]:
runs = load_recent_runs(limit=50)
print(f"{len(runs)} run records found in datasets/reports/runs/")

rows = []
for run in runs:
    for stage in run["stages"]:
        rows.append({
            "run_name": run["run_name"],
            "stage": stage["name"],
            "status": stage["status"],
            "duration_s": stage["duration_s"],
            **stage.get("metrics", {}),
        })

df = pd.DataFrame(rows)
df

28 run records found in datasets/reports/runs/


,run_name,stage,status,duration_s,source,bucket,prefix,files_downloaded,sources_found,chip_id,...,infer_chips_per_sec,events,clean_iou,n_nodata_s1_pixels,noisy_iou,iou_drop,connected_to_aws_braket,feature_channels,n_valid_pixels,water_fraction
0,01_ingest,ingest_sen1floods11,ok,2.620,s3,sphoorthq-geoverse,datasets/raw/sen1floods11/,2235.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,01_ingest,catalog_local_raw,ok,0.013,NaN,NaN,NaN,NaN,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,01_ingest,ingest_sen1floods11,ok,3.135,s3,sphoorthq-geoverse,datasets/raw/sen1floods11/,2235.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,01_ingest,catalog_local_raw,ok,0.000,NaN,NaN,NaN,NaN,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,02_process,load_chip,ok,0.168,NaN,NaN,NaN,NaN,NaN,Ghana_103272,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
93,03_feature_engineering,flatten_to_pixel_table,ok,0.008,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,259269.0,0.006
94,02_process,load_chip,ok,0.071,NaN,NaN,NaN,NaN,NaN,Ghana_103272,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
95,02_process,speckle_filter,ok,0.036,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
96,01_ingest,verify_sen1floods11_source,ok,1.419,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
if not df.empty:
    print("Total stage time per notebook run:")
    print(df.groupby("run_name")["duration_s"].sum().round(2))
    print()
    print("Any failed stages:")
    failed = df[df["status"] == "failed"]
    print(failed if not failed.empty else "none")

Total stage time per notebook run:
run_name
01_ingest                          16.33
02_process                          0.47
03_feature_engineering              0.06
04_classical_ml                  2398.02
05_qml_ibm                         52.37
05_qml_ibm_braket                 649.77
06_hybrid_ensemble_evaluation     835.72
08_robustness_sweep               103.02
09_patch_unet                     488.00
10_quantum_enhanced_unet           66.54
Name: duration_s, dtype: float64

Any failed stages:
none
